# Pipeline RAG completo — Regulamento Geral da Graduação da UFRPE

Este é o notebook principal. O corpus é o **Regulamento Geral da Graduação (PREG/UFRPE, 2024)**, dividido em 30 documentos temáticos por `split_regulamento.py` (um ou mais Títulos/Capítulos por documento, por exemplo matrícula, suspensão de programa, monitoria e glossário). O notebook lê os PDFs de `documents/`, cria chunks de tamanho fixo com overlap, persiste embeddings no ChromaDB e compara os dois métodos de retrieval:

- **esparso:** TF-IDF com similaridade cosseno;
- **denso:** embeddings Gemini com similaridade cosseno no ChromaDB.

Ao final, três perguntas são respondidas sem retrieval e com os dois retrievers. Uma pergunta demonstrativa adicional completa as 10 respostas da avaliação manual.

Para regenerar o corpus: `python split_regulamento.py "Regulamento geral PREG 2024.pdf" documents`.

## 1. Modelos, caminhos e banco

In [ ]:
import os
from pathlib import Path
from pprint import pprint

import chromadb
from dotenv import load_dotenv
from google import genai

import rag_utils

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "documents"
CHROMA_DIR = PROJECT_ROOT / "db" / "chroma"
COLLECTION_NAME = "regulamento_ufrpe_v1"
EMBED_MODEL = "gemini-embedding-001"
GEN_MODEL = "gemini-3.1-flash-lite"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 5

load_dotenv(PROJECT_ROOT / ".env")
API_KEY = os.environ.get("GEMINI_API_KEY")
assert API_KEY, "Defina GEMINI_API_KEY no arquivo .env da raiz do projeto."

client = genai.Client(api_key=API_KEY)
embedder = rag_utils.GeminiEmbedder(client, model=EMBED_MODEL)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"}
)
print(f"Documentos: {DATA_DIR}")
print(f"Banco: {CHROMA_DIR}")

## 2. Ingestão e indexação

A indexação é retomável: chunks já presentes no banco não são enviados novamente à API de embeddings.

In [ ]:
pdf_paths = rag_utils.iter_pdfs(DATA_DIR)
records = rag_utils.load_corpus(DATA_DIR, CHUNK_SIZE, CHUNK_OVERLAP)
assert records, f"Nenhum conteúdo PDF foi encontrado em {DATA_DIR}."

index_stats = rag_utils.index_dense_collection(collection, records, embedder)
print(f"PDFs: {len(pdf_paths)}")
print(f"Chunks: {len(records)}")
print(index_stats)

## 3. Os dois métodos de retrieval

Ambos usam exatamente os mesmos chunks. O método denso consulta os vetores persistidos no Chroma; o TF-IDF monta um índice esparso local a partir dos textos guardados no mesmo banco.

In [ ]:
indexed_records = rag_utils.records_from_collection(collection)
dense_retriever = rag_utils.DenseRetriever(collection, embedder)
tfidf_retriever = rag_utils.TfidfRetriever(indexed_records)

dense_rag = rag_utils.RAGAssistant(client, dense_retriever, GEN_MODEL)
tfidf_rag = rag_utils.RAGAssistant(client, tfidf_retriever, GEN_MODEL)

In [ ]:
sample_question = "Por quanto tempo um(a) discente pode manter a condição de aluno especial?"

print("Ranking denso:")
pprint(rag_utils.hits_as_rows(dense_retriever.retrieve(sample_question, TOP_K)), sort_dicts=False)
print("\nRanking TF-IDF:")
pprint(rag_utils.hits_as_rows(tfidf_retriever.retrieve(sample_question, TOP_K)), sort_dicts=False)

## 4. Geração fundamentada

O prompt contém os top-k chunks completos e exige citações no formato `[documento, p. N, trecho M]`.

In [ ]:
demo_result = dense_rag.ask(sample_question, k=TOP_K)
print(demo_result.answer)

## 5. Três perguntas: sem retrieval, RAG denso e RAG TF-IDF

Esta célula gera nove respostas comparáveis. Somadas à demonstração acima, são as 10 respostas usadas na avaliação manual.

In [ ]:
QUESTIONS = [
    "Qual é o limite máximo de suspensões de programa e até quando a suspensão deve ser solicitada?",
    "Em que situação um(a) discente é colocado(a) no regime de observação do desempenho acadêmico?",
    "Como é calculada a nota final da seleção para monitoria?",
]

generated_answers = [
    {"question": sample_question, "method": "dense", "answer": demo_result.answer}
]

for question in QUESTIONS:
    baseline = rag_utils.answer_without_retrieval(client, question, GEN_MODEL)
    dense_result = dense_rag.ask(question, k=TOP_K)
    tfidf_result = tfidf_rag.ask(question, k=TOP_K)

    generated_answers.extend([
        {"question": question, "method": "no_retrieval", "answer": baseline},
        {"question": question, "method": "dense", "answer": dense_result.answer},
        {"question": question, "method": "tfidf", "answer": tfidf_result.answer},
    ])

    print(f"\n{'=' * 90}\nPERGUNTA: {question}")
    print(f"\nSEM RETRIEVAL\n{baseline}")
    print("\nCHUNKS — DENSO")
    pprint(rag_utils.hits_as_rows(dense_result.hits), sort_dicts=False)
    print(f"\nRAG DENSO\n{dense_result.answer}")
    print("\nCHUNKS — TF-IDF")
    pprint(rag_utils.hits_as_rows(tfidf_result.hits), sort_dicts=False)
    print(f"\nRAG TF-IDF\n{tfidf_result.answer}")

assert len(generated_answers) == 10

## 6. Avaliação manual das 10 respostas

Preencha `True`/`False` depois de ler cada resposta e, nos casos com retrieval, os chunks exibidos. Para a baseline sem retrieval, suporte e citação não se aplicam.

In [ ]:
evaluation_sheet = [
    {
        "answer_id": index,
        "question": item["question"],
        "method": item["method"],
        "correct": None,
        "supported_by_chunks": None if item["method"] != "no_retrieval" else "N/A",
        "cites_right_chunk": None if item["method"] != "no_retrieval" else "N/A",
        "notes": "",
    }
    for index, item in enumerate(generated_answers, start=1)
]
evaluation_sheet

## 7. Diagnóstico de duas respostas erradas

Depois da avaliação, escolha dois `answer_id` incorretos. Marque **retrieval** se os chunks necessários não chegaram ao modelo; marque **generation** se os chunks eram adequados, mas a resposta os interpretou mal, ignorou evidências ou citou o trecho errado.

In [ ]:
failure_analysis = [
    {"answer_id": None, "failure_source": "retrieval ou generation", "justification": ""},
    {"answer_id": None, "failure_source": "retrieval ou generation", "justification": ""},
]
failure_analysis

## 8. Risco de busca responsável

**Atualidade (freshness):** o índice é uma fotografia do Regulamento Geral da Graduação de 2024 no momento da ingestão. O próprio regulamento já traz dispositivos alterados por resoluções posteriores ("redação dada pela Resolução..."). Se o CEPE aprovar uma nova resolução mudando, por exemplo, o limite de suspensões de programa ou os critérios do RODA, o sistema continuará respondendo a regra antiga com citação e aparência de autoridade, e um(a) discente pode perder um prazo ou tomar uma decisão acadêmica errada.

**Mitigação:** registrar no metadado de cada chunk a versão e a data do regulamento, mostrar essa data junto de toda resposta ("segundo o Regulamento PREG 2024") e reindexar automaticamente quando a PREG publicar uma nova versão ou resolução, removendo os chunks do documento substituído. Para decisões com prazo, a resposta deve recomendar a confirmação com a coordenação ou o DRCA.